# Outlet Framing Analysis

Comparing frame usage between individual outlets and by political valence (Left, Center, Right).

In [ ]:
!pip install pandas numpy matplotlib seaborn pyarrow scipy

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from matplotlib.colors import LinearSegmentedColormap

In [ ]:
df = pd.read_parquet('data/merged_topic_and_frames_filtered.parquet')
print(f'Loaded {len(df):,} articles')

frames = ['economic', 'fairness', 'public_op', 'political', 'quality_life', 
          'crime', 'culture', 'health', 'legality', 'morality', 
          'policy', 'regulation', 'security', 'cap&res']

frame_matrix = np.array(df['vector'].tolist())
for i, frame in enumerate(frames):
    df[f'frame_{frame}'] = frame_matrix[:, i]

frame_cols = [f'frame_{f}' for f in frames]

valence_map = {'Left': 'Left', 'Lean Left': 'Left', 'Center': 'Center', 'Lean Right': 'Right', 'Right': 'Right'}
df['valence'] = df['bias'].map(valence_map)

print(f'\nArticles by valence:')
print(df['valence'].value_counts())

## Frame Usage by Individual Outlet

In [ ]:
outlet_frames = df.groupby('outlet_name')[frame_cols].mean()
outlet_frames.columns = frames
outlet_frames['n_articles'] = df.groupby('outlet_name').size()
outlet_frames = outlet_frames.sort_values('n_articles', ascending=False)

outlet_frames.style.format('{:.1%}', subset=frames).format('{:,}', subset=['n_articles'])

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(outlet_frames[frames], annot=True, fmt='.0%', cmap='YlOrRd', ax=ax)
plt.title('Frame Usage Rate by Outlet')
plt.tight_layout()
plt.show()

In [ ]:
print('Frame usage range across outlets (max - min):\n')
for frame in frames:
    vals = outlet_frames[frame]
    print(f'{frame:15} {vals.min():.0%} - {vals.max():.0%}  (range: {vals.max()-vals.min():.0%})')

## Frame Usage by Political Valence

In [ ]:
valence_frames = df.groupby('valence')[frame_cols].mean()
valence_frames.columns = frames
valence_frames['n_articles'] = df.groupby('valence').size()
valence_frames = valence_frames.loc[['Left', 'Center', 'Right']]

valence_frames.style.format('{:.1%}', subset=frames).format('{:,}', subset=['n_articles'])

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
valence_frames[frames].T.plot(kind='bar', ax=ax)
plt.title('Frame Usage by Political Valence')
plt.xlabel('Frame')
plt.ylabel('Usage Rate')
plt.legend(title='Valence')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# custom colormap to avoid red/blue political connotations
colors = ['#8B4513', '#FFFFFF', '#228B22']  # brown -> white -> green
cmap_neutral = LinearSegmentedColormap.from_list('BrownGreen', colors)

diff = valence_frames.loc['Left', frames] - valence_frames.loc['Right', frames]
diff_df = pd.DataFrame({'left_minus_right': diff}).sort_values('left_minus_right', key=abs, ascending=False)

print('Left vs Right frame usage difference:')
print('(positive/green = Left uses more, negative/brown = Right uses more)\n')
diff_df.style.format('{:+.1%}').background_gradient(cmap=cmap_neutral, vmin=-0.2, vmax=0.2)

## Statistical Significance: Chi-Square Tests

In [ ]:
chi2_results = []
for frame in frames:
    col = f'frame_{frame}'
    contingency = pd.crosstab(df['valence'], df[col])
    chi2, p, dof, expected = stats.chi2_contingency(contingency)
    chi2_results.append({'frame': frame, 'chi2': chi2, 'p_value': p, 'significant': p < 0.05})

chi2_df = pd.DataFrame(chi2_results).sort_values('chi2', ascending=False)
chi2_df.style.format({'chi2': '{:.1f}', 'p_value': '{:.2e}'})

## Frame Usage by Valence and Topic

In [ ]:
topics = df['topic_top1'].value_counts().index.tolist()
print('Topics:', topics)

In [ ]:
def valence_comparison_for_topic(topic):
    subset = df[df['topic_top1'] == topic]
    valence_topic = subset.groupby('valence')[frame_cols].mean()
    valence_topic.columns = frames
    valence_topic = valence_topic.loc[['Left', 'Center', 'Right']]
    diff = valence_topic.loc['Left'] - valence_topic.loc['Right']
    return valence_topic, diff

for topic in topics[:4]:
    valence_topic, diff = valence_comparison_for_topic(topic)
    n = len(df[df['topic_top1'] == topic])
    print(f'\n=== {topic.upper()} (n={n:,}) ===')
    print('\nFrame usage by valence:')
    display(valence_topic.style.format('{:.1%}'))
    print('\nLargest Left-Right differences:')
    top_diff = diff.abs().sort_values(ascending=False).head(5)
    for frame in top_diff.index:
        print(f'  {frame}: {diff[frame]:+.1%}')

In [ ]:
diff_by_topic = []
for topic in topics:
    _, diff = valence_comparison_for_topic(topic)
    diff_by_topic.append(diff)

diff_matrix = pd.DataFrame(diff_by_topic, index=topics)

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(diff_matrix, annot=True, fmt='+.0%', cmap=cmap_neutral, center=0, ax=ax, vmin=-0.25, vmax=0.25)
plt.title('Left vs Right Frame Usage Difference by Topic\n(positive/green = Left uses more, negative/brown = Right uses more)')
plt.tight_layout()
plt.show()

## Frame Trends Over Time by Outlet

In [ ]:
df['date'] = pd.to_datetime(df['date'])
df['quarter'] = df['date'].dt.to_period('Q')

outlet_trends = []
for outlet in df['outlet_name'].unique():
    subset = df[df['outlet_name'] == outlet]
    quarterly = subset.groupby('quarter')[frame_cols].mean()
    x = np.arange(len(quarterly))

    for frame_col in frame_cols:
        y = quarterly[frame_col].values
        slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
        outlet_trends.append({'outlet': outlet, 'frame': frame_col.replace('frame_', ''), 'slope_per_year': slope * 4, 'p_value': p_value})

outlet_trend_df = pd.DataFrame(outlet_trends)
outlet_trend_pivot = outlet_trend_df.pivot(index='outlet', columns='frame', values='slope_per_year')
outlet_trend_pivot = outlet_trend_pivot[frames]
outlet_trend_pivot.style.format('{:+.1%}').background_gradient(cmap='PiYG', vmin=-0.05, vmax=0.05)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(outlet_trend_pivot, annot=True, fmt='+.0%', cmap='PiYG', center=0, ax=ax)
plt.title('Frame Usage Trend by Outlet (Annual % Change)')
plt.tight_layout()
plt.show()

## Frame Trends Over Time by Political Valence

In [ ]:
valence_trends = []
for valence in ['Left', 'Center', 'Right']:
    subset = df[df['valence'] == valence]
    quarterly = subset.groupby('quarter')[frame_cols].mean()
    x = np.arange(len(quarterly))

    for frame_col in frame_cols:
        y = quarterly[frame_col].values
        slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
        valence_trends.append({'valence': valence, 'frame': frame_col.replace('frame_', ''), 'slope_per_year': slope * 4, 'p_value': p_value})

valence_trend_df = pd.DataFrame(valence_trends)
valence_trend_pivot = valence_trend_df.pivot(index='valence', columns='frame', values='slope_per_year')
valence_trend_pivot = valence_trend_pivot.loc[['Left', 'Center', 'Right'], frames]
valence_trend_pivot.style.format('{:+.1%}').background_gradient(cmap='PiYG', vmin=-0.03, vmax=0.03)

In [ ]:
key_frames = ['fairness', 'culture', 'legality', 'public_op']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for idx, frame in enumerate(key_frames):
    frame_col = f'frame_{frame}'
    ax = axes[idx]

    for valence in ['Left', 'Center', 'Right']:
        subset = df[df['valence'] == valence]
        quarterly = subset.groupby('quarter')[frame_col].mean()
        ax.plot(quarterly.index.astype(str), quarterly.values, label=valence, marker='o', markersize=3)

    ax.set_title(f'{frame.title()} Frame')
    ax.set_xlabel('Quarter')
    ax.set_ylabel('Usage Rate')
    ax.tick_params(axis='x', rotation=45)

axes[0].legend(title='Valence')
plt.suptitle('Frame Usage Over Time by Political Valence', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# check if valences are converging or diverging
quarterly_gap = []
for quarter in sorted(df['quarter'].unique()):
    q_df = df[df['quarter'] == quarter]
    left_mean = q_df[q_df['valence'] == 'Left'][frame_cols].mean()
    right_mean = q_df[q_df['valence'] == 'Right'][frame_cols].mean()
    gap = (left_mean - right_mean).abs().mean()
    quarterly_gap.append({'quarter': quarter, 'avg_gap': gap})

gap_df = pd.DataFrame(quarterly_gap).sort_values('quarter')
x = np.arange(len(gap_df))
y = gap_df['avg_gap'].values
slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(gap_df['quarter'].astype(str), gap_df['avg_gap'], marker='o')
ax.plot(gap_df['quarter'].astype(str), intercept + slope * x, 'r--', label=f'Trend: {slope*4:+.2%}/year')
plt.title('Average Left-Right Frame Usage Gap Over Time')
plt.xlabel('Quarter')
plt.ylabel('Average Absolute Difference')
plt.xticks(rotation=45, ha='right')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Gap trend: {slope*4:+.2%} per year (p={p_value:.4f})')
if slope > 0:
    print('Left and Right outlets are diverging in their framing over time')
else:
    print('Left and Right outlets are converging in their framing over time')